# Урок 7.4. Альтернативы: LangGraph, AutoGen и критерии выбора

Финал модуля 7. У нас есть три системы координат: свой оркестратор (7.2, полный
контроль), CrewAI (7.3, декларативные роли). Сегодня добавим ещё две — **графовую**
(LangGraph) и **диалоговую** (AutoGen), обе живьём на нашей qwen2.5:3b, — и
соберём итоговую таблицу выбора. Закроем модуль сводным квизом и финальной
практикой.

🎯 **В этом уроке:**

- Перенести конечный автомат из 7.2 на LangGraph — и увидеть, что это буквально он.
- Запустить диалоговую команду AutoGen и понять, чем «разговор» отличается от «конвейера».
- Свести все четыре подхода в одну таблицу с критериями выбора.
- Сводный квиз модуля и финальная практика.

**Что понадобится:** Ollama с qwen2.5:3b и окружение `.venv` модуля — в нём,
кроме CrewAI, теперь живут `langgraph`, `langchain-ollama` и `autogen-agentchat`
(обновите: `pip install -r requirements.txt`).

## 1. Четыре парадигмы — одна работа

| Парадигма | Кто | Оркестратор — это... | Вы описываете... |
|---|---|---|---|
| императивная | свой код (7.2) | ваш Python | каждый шаг руками |
| декларативная | CrewAI (7.3) | скрытый цикл фреймворка | роли и задачи |
| **графовая** | **LangGraph** | явный граф состояний | узлы, рёбра, маршруты |
| **диалоговая** | **AutoGen** | групповой чат | участников и правило остановки |

Все четыре решают одно и то же: кто говорит следующим и когда остановиться.
Разница — в том, где живёт эта логика и насколько она вам видна. Дальше — обе
новые парадигмы руками.

Две ремарки перед установкой (обе — из живого опыта на этой машине):

- **Windows может заблокировать нативные DLL свежих пакетов** («Политика
  управления приложениями заблокировала этот файл» при импорте). Лечение:
  `Get-ChildItem .venv -Recurse -Filter *.pyd | Unblock-File` в PowerShell —
  pip проставляет скачанным файлам «метку из интернета».
- **Фреймворки конфликтуют в одном окружении.** AutoGen при установке откатил
  `protobuf` до 5.x — и сломал импорт CrewAI (его телеметрии нужен 6.x+).
  Лечение здесь — вернуть новый protobuf (runtime новее генкода допустим);
  урок — зоопарк агентных фреймворков лучше держать в разных клетках-venv.

## 2. LangGraph: наш конечный автомат, буквально

В уроке 3.6 LangGraph управлял одним агентом с инструментами. Для команды
идея та же: **состояние** (наша доска с кодом и отчётом), **узлы** (роли и
судья-тесты) и **рёбра с условиями** (политика оркестратора). Диаграмма из
урока 7.2 перекладывается в код почти дословно — сравните сами.

In [1]:
import json
import re
import subprocess
import sys
import tempfile
from pathlib import Path
from typing import TypedDict

from langchain_ollama import ChatOllama
from langgraph.graph import END, START, StateGraph

llm = ChatOllama(model="qwen2.5:3b", base_url="http://127.0.0.1:11434",
                 temperature=0, num_predict=600)

# --- задача и судья: компактная версия харнеса из 7.2 ---------------------------
TASK = """Функция Python normalize_phone(text: str) -> str: приводит российский
номер телефона к каноничному виду - строка из 11 цифр БЕЗ разделителей,
начинается с 7. Номер, начинающийся с 8, - это тот же номер с кодом 7.
Если корректный номер не получается - подними ValueError."""

TESTS = [
    ('normalize_phone("+7 (912) 345-67-89")', "79123456789"),
    ('normalize_phone("8-912-345-67-89")', "79123456789"),
    ('normalize_phone("79123456789")', "79123456789"),
]
ERRORS = ['normalize_phone("12345")']

HARNESS = """
import json
results = []
{code}
for expr, expected in {tests!r}:
    try:
        got = eval(expr)
        results.append({{"case": expr, "ok": got == expected, "got": repr(got), "expected": expected}})
    except Exception as exc:
        results.append({{"case": expr, "ok": False, "got": f"{{type(exc).__name__}}: {{exc}}", "expected": expected}})
for expr in {errors!r}:
    try:
        eval(expr); results.append({{"case": expr, "ok": False, "got": "нет исключения", "expected": "ValueError"}})
    except ValueError:
        results.append({{"case": expr, "ok": True, "got": "ValueError", "expected": "ValueError"}})
    except Exception as exc:
        results.append({{"case": expr, "ok": False, "got": type(exc).__name__, "expected": "ValueError"}})
print(json.dumps(results, ensure_ascii=False))
"""

def run_tests(code_str: str) -> list[dict]:
    src = HARNESS.format(code=code_str, tests=TESTS, errors=ERRORS)
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False, encoding="utf-8") as f:
        f.write(src)
        path = f.name
    try:
        proc = subprocess.run([sys.executable, "-X", "utf8", path], capture_output=True,
                              text=True, encoding="utf-8", timeout=10)
        if proc.returncode != 0:
            return [{"case": "<загрузка>", "ok": False, "got": proc.stderr[-150:], "expected": "импорт"}]
        return json.loads(proc.stdout)
    finally:
        Path(path).unlink(missing_ok=True)

print(f"судья готов: {len(TESTS) + len(ERRORS)} кейсов")

судья готов: 4 кейсов


In [2]:
# --- состояние, узлы, рёбра ------------------------------------------------------
class TeamState(TypedDict):
    task: str
    code: str
    report: str
    passed: bool
    fix_rounds: int


def developer(state: TeamState) -> dict:
    """Узел-роль: пишет код (или чинит по отчёту). Возвращает ДЕЛЬТУ состояния."""
    if state.get("report"):
        prompt = (f"Задача:\n{state['task']}\n\nТвой код не прошёл тесты:\n"
                  f"```python\n{state['code']}\n```\nОтчёт:\n{state['report']}\n\n"
                  "Исправь функцию. Верни ТОЛЬКО код одним блоком ```python```.")
    else:
        prompt = (f"Задача:\n{state['task']}\n\n"
                  "Напиши функцию. Верни ТОЛЬКО код одним блоком ```python```.")
    answer = llm.invoke(prompt).content
    match = re.search(r"```(?:python)?\s*(.+?)```", answer, re.DOTALL)
    code_str = match.group(1).strip() if match else state.get("code", "")
    print(f"  [developer] попытка {state.get('fix_rounds', 0) + 1}")
    return {"code": code_str, "fix_rounds": state.get("fix_rounds", 0) + 1}


def tester(state: TeamState) -> dict:
    """Узел-судья: обычный код, никакой LLM."""
    results = run_tests(state["code"])
    report = "\n".join(f"[{'PASS' if r['ok'] else 'FAIL'}] {r['case']} -> {r['got']}"
                        f" (ожидалось {r['expected']})" for r in results)
    print(f"  [tester] {sum(r['ok'] for r in results)}/{len(results)}")
    return {"report": report, "passed": all(r["ok"] for r in results)}


def route(state: TeamState) -> str:
    """Политика оркестратора - в чистом виде, как route() из урока 3.6."""
    if state["passed"]:
        return "accept"
    if state["fix_rounds"] >= 3:
        return "give_up"     # лимит - в коде, как всегда
    return "retry"


graph = StateGraph(TeamState)
graph.add_node("developer", developer)
graph.add_node("tester", tester)
graph.add_edge(START, "developer")
graph.add_edge("developer", "tester")
graph.add_conditional_edges("tester", route,
                            {"accept": END, "give_up": END, "retry": "developer"})
app = graph.compile()

print(app.get_graph().draw_ascii())   # pip install grandalf для ASCII-схемы

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
+-----------+  
| developer |  
+-----------+  
      .        
      .        
      .        
  +--------+   
  | tester |   
  +--------+   
      .        
      .        
      .        
 +---------+   
 | __end__ |   
 +---------+   


In [3]:
final = app.invoke({"task": TASK, "code": "", "report": "",
                    "passed": False, "fix_rounds": 0})
print("\nВердикт:", "ПРИНЯТО" if final["passed"] else "нужен человек",
      f"({final['fix_rounds']} попыток)")
print(final["code"])

  [developer] попытка 1
  [tester] 2/4


  [developer] попытка 2
  [tester] 2/4


  [developer] попытка 3
  [tester] 2/4

Вердикт: нужен человек (3 попыток)
def normalize_phone(text: str) -> str:
    if not text.isdigit() or len(text) != 11:
        raise ValueError("Некорректный номер телефона")
    if text.startswith('8'):
        text = '7' + text[1:]
    return text[:-1] if text[-1] == '8' else text


Разбор моего прогона:

- **Изоморфизм полный.** `route()` — это политика оркестратора из 7.2 в чистом
  виде; `tester` — узел из обычного кода (LLM в графе не обязана быть в каждом
  узле!); лимит — счётчик в состоянии. Диаграмма, которую мы рисовали ASCII-артом
  в 7.2, теперь рисуется сама — `draw_ascii()` выше.
- **Сходимость не изменилась: 3 попытки, 2/4, «нужен человек».** Переезд на
  фреймворк не чинит модель: сходимость фикс-цикла — свойство модели и читаемости
  отчёта, а не синтаксиса оркестрации. Полюбуйтесь на финальный код: строка
  «отрезать последнюю цифру, если это 8» — анкоринг в новом обличье (модель
  переиначила правило про *первую* восьмёрку).
- **Что LangGraph даёт сверх наших 150 строк:** `checkpointer` (MemorySaver из
  урока 3.6 — команда переживает перезапуск процесса и продолжает с места
  остановки), `interrupt` (пауза на человека ДО эскалации — human-in-the-loop
  без самодельных костылей), стриминг событий узлов и картинка графа.
- Версии: урок 3.6 мы писали под langgraph 0.2–0.4, сейчас в venv стоит 1.x —
  и весь использованный там словарь (`StateGraph`, `START`/`END`,
  `add_conditional_edges`, дельты состояния) работает без правок. База у
  фреймворка устойчивая, пины в requirements всё равно оставляем.

## 3. AutoGen: команда как разговор

AutoGen (сегодня это `autogen-agentchat` от Microsoft; форк старого API живёт
под именем AG2) исповедует другую веру: агенты не передают артефакты по
конвейеру, а **разговаривают в общем чате**. Оркестратор — это правило, кто
говорит следующим (`RoundRobinGroupChat` — по кругу, `SelectorGroupChat` —
выбирает LLM), а остановка — **условие над текстом чата**: прозвучало
стоп-слово, кончился лимит сообщений.

Сценарий для пробы — не код (для кода мы уже выбрали конвейер с судьёй), а
задача, где диалог уместен: писатель сочиняет уведомление для клиентов,
критик проверяет по чеклисту и либо возвращает на доработку, либо говорит
«ОДОБРЕНО» — и это слово останавливает чат.

In [4]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_ext.models.ollama import OllamaChatCompletionClient

client = OllamaChatCompletionClient(
    model="qwen2.5:3b", host="http://127.0.0.1:11434",
    options={"temperature": 0, "num_predict": 300},
)

writer = AssistantAgent(
    "writer", model_client=client,
    system_message="Ты пишешь короткие уведомления для клиентов хостинга "
                   "«Пингвин.Хост». Одно уведомление, 2-3 предложения.")
critic = AssistantAgent(
    "critic", model_client=client,
    system_message="Ты — критик. Проверь уведомление: есть ли конкретное время "
                   "работ и извинение за неудобства. Если всё есть - ответь ровно "
                   "ОДОБРЕНО. Если нет - скажи, что добавить.")

team = RoundRobinGroupChat(
    [writer, critic],
    termination_condition=TextMentionTermination("ОДОБРЕНО")   # стоп-слово...
                          | MaxMessageTermination(7),          # ...и стоп-кран
)

# в Jupyter - await (как kickoff_async в уроке 7.3); в скрипте - asyncio.run(...)
result = await team.run(
    task="Напиши уведомление о плановых работах в ночь на субботу с 02:00 до 04:00 МСК.")

for message in result.messages:
    print(f"--- {getattr(message, 'source', '?')}:")
    print(str(getattr(message, 'content', ''))[:300], "\n")
print("остановка:", result.stop_reason)

--- user:
Напиши уведомление о плановых работах в ночь на субботу с 02:00 до 04:00 МСК. 

--- writer:
Внимание: В период с 02:00 до 04:00 МСК на субботу будут проводиться плановые технические работы. 

--- critic:
ОДОБРЕНО. 

остановка: Text 'ОДОБРЕНО' mentioned


Диалог сошёлся за два хода: писатель написал, критик сказал стоп-слово. Уютно.
А теперь перечитайте уведомление: **извинения за неудобства в нём нет** — а
критик обязан был его требовать по собственной инструкции. 3B-критик с чеклистом
«в голове» пропустил пункт и одобрил. Это ровно урок 7.1: свободное решение
маленькой модели ненадёжно, работает только жёсткая форма — но в диалоговой
парадигме некуда воткнуть наш `run_tests` между репликами: проверка выхода здесь —
тоже просто участник разговора, со всеми его слабостями.

Вторая хрупкость — терминация. `TextMentionTermination("ОДОБРЕНО")` ищет
**подстроку в тексте чата**: ответь критик «НЕ ОДОБРЕНО» — чат остановился бы
с тем же успехом. Поэтому стоп-кран `MaxMessageTermination` — не опция, а
обязательный страхующий, и поэтому серьёзные решения в диалоговых системах
всё равно выносят наружу, в код.

Когда диалоговая парадигма уместна: брейншторм и генерация вариантов, интервью
и уточнение требований, доработка текста с человеком в цикле — всё, где ценен
сам ход разговора, а не гарантированный артефакт на выходе. Наш кейс с
уведомлением — пограничный: для прода мы бы вынесли чеклист критика в код
(пять якорей стиля из модуля 6 — готовый образец).

## 4. Итоговая таблица и критерии выбора

| | свой код (7.2) | CrewAI (7.3) | LangGraph | AutoGen |
|---|---|---|---|---|
| парадигма | императивная | декларативная | графовая | диалоговая |
| оркестратор | ваш Python | скрытый цикл | явный граф | правило чата |
| судья кодом | встроили сами | сами поверх | узел графа — естественно | сами поверх |
| циклы и лимиты | любые, в коде | ограниченно | рёбра + счётчики в состоянии | лимит сообщений |
| маршрут виден | да (это ваш код) | нет | да (граф рисуется) | нет (решает чат) |
| персистентность | сами | память crew | checkpointer из коробки (3.6) | сериализация чата |
| порог входа | 150 строк | 40 строк | ~60 строк + концепции | ~30 строк |
| зависимости | httpx | десятки | langgraph + langchain-core | autogen-* |
| на локальной 3B | предсказуемо | англ. обвязка | предсказуемо (промпты ваши) | терминация хрупкая |

Критерии выбора — сверху вниз:

1. **Конвейер известен заранее и есть судья-код** → свой оркестратор или
   LangGraph (граф даёт бесплатно checkpointer, стриминг и картинку — берите его,
   когда конвейер переживёт этот ноутбук).
2. **Нужна экосистема**: готовые инструменты, интеграции, команда на общем
   словаре → CrewAI.
3. **Задача — действительно разговор**: брейншторм, интервью, доработка текста
   с человеком в цикле → AutoGen.
4. **Ничего из перечисленного** → вернитесь к уроку 7.1: возможно, вам нужен
   один агент. Или ноль.

И общий для всех четырёх закон модуля: **фреймворк не судит и не ограничивает —
судья, лимиты и ветка «нужен человек» всегда ваши.**

## ⚠️ Частые ошибки

1. **Терминация по подстроке.** `TextMentionTermination("ОДОБРЕНО")` сработает
   и на «НЕ ОДОБРЕНО» — это поиск подстроки в тексте чата. Стоп-слово должно
   быть таким, чтобы не встречалось в отрицаниях, а стоп-кран по числу сообщений —
   обязателен всегда.
2. **Зоопарк фреймворков в одном venv.** Наш живой пример: AutoGen откатил
   protobuf и сломал CrewAI. В работе — по фреймворку на окружение.
3. **Выбор фреймворка раньше выбора задачи.** Сначала конвейер/судья/лимиты
   на бумаге (урок 7.2), потом — во что это дешевле всего одеть.
4. **Диалоговая парадигма для конвейера.** Если маршрут известен, чат — это
   способ потратить токены на вежливость между этапами.
5. **Вера, что «через полгода будет так же».** langgraph 0.2 из урока 3.6 стал
   1.x к этому уроку (базовые API устояли, но пины в requirements — не паранойя,
   а гигиена).
6. **Забыть про Windows-политику на свежие DLL.** «DLL load failed... политика
   заблокировала файл» после pip install — это не вирус и не битый пакет:
   снимите метку интернета `Unblock-File` (см. README модуля).

## Сводный квиз модуля 7

**1. «Сделай три роли в одном промпте» — какие две вещи сломаются первыми (урок 7.1)?**

<details><summary>Ответ</summary>

Бюджет (один num_predict на всех — обрыв на финальном артефакте,
done_reason=length) и само-ревью (нет дистанции: модель хвалит свой текст и
находит выдуманные проблемы). Роли расходятся по отдельным вызовам именно за этим.
</details>

**2. Наивный судья скопировал уверенный вердикт вместо решения. Какой приём это чинит и почему он работает?**

<details><summary>Ответ</summary>

Чеклист — жёсткий формат, который заставляет сначала выписать проверяемые факты
(пустые списки!) и только потом выбрать вердикт по правилу. Маленькой модели
нельзя доверить свободное взвешивание, но можно — заполнение формы (CoT из 1.7
в структуре).
</details>

**3. Чем судья-тесты принципиально лучше судьи-LLM для кода — три причины из урока 7.2?**

<details><summary>Ответ</summary>

Его нельзя убедить текстом; он бесплатный (0 токенов на вердикт); его отчёт —
объективный вход для следующего раунда. LLM-ревью остаётся для непроверяемого
кодом (стиль, читаемость).
</details>

**4. Фикс-цикл трижды меняет мелочи, тесты красные. Назовите болезнь и два лекарства оркестратора.**

<details><summary>Ответ</summary>

Анкоринг на собственном коде. Лекарства: эскалация с чистого листа (диагноз
передать, старый код спрятать) и — если модальный ответ модели пережил и это —
честный REJECTED с журналом для человека. Оба решения принимает код по счётчикам.
</details>

**5. Почему отчёт тестов «AssertionError, кейс 3» гарантирует бесконечный фикс-цикл на 3B?**

<details><summary>Ответ</summary>

Ревьюер-LLM чинит по отчёту, а такой отчёт не содержит ни входа, ни полученного,
ни ожидаемого значения — причина невидима, модель гадает. Отчёт — интерфейс:
вход/got/expected обязательны (наш parse_size не сошёлся ни разу именно из-за
непрозрачных падений).
</details>

**6. `expected_output` в CrewAI и приёмочные тесты — в чём разница по существу?**

<details><summary>Ответ</summary>

`expected_output` — текст-пожелание, который фреймворк вставляет в промпт и никогда
не проверяет. Приёмочные тесты — исполняемая проверка с вердиктом. Панель «Task
Completed» при 2/6 на тестах — живая иллюстрация разницы.
</details>

**7. Чем supervisor-паттерн отличается от pipeline и когда LLM-супервизор — лишний?**

<details><summary>Ответ</summary>

Pipeline — маршрут известен и зашит; supervisor — LLM разбирает задачу на
подзадачи по входу. Если декомпозиция всегда одинаковая, супервизор-LLM — плата
за то, что код сделал бы бесплатно и надёжно.
</details>

**8. Почему «прогнать crew в цикле с отчётом тестов» (упр. 7.3) — типичный финал взросления системы?**

<details><summary>Ответ</summary>

Потому что судья, ретраи и лимиты — ответственность архитектора, и они живут
в вашем коде. Фреймворк оказывается деталью внутри вашего оркестратора: он
удобно исполняет роли, а решения принимает ваш цикл.
</details>

**9. Команда из 3 ролей подняла качество с 82% до 85%, латентность — с 4 до 14 секунд. Ваши действия?**

<details><summary>Ответ</summary>

Считать по профилю нагрузки (урок 5.1): для фонового пайплайна +3 п.п. может
окупиться; для интерактива — почти никогда. И проверить лестницу подешевле:
промпт, RAG, грамматика, дообучение (модуль 6) — прежде чем платить ×3 за каждый
запрос вечно.
</details>

**10. Какие три вещи обязаны быть «в коде», какой бы фреймворк вы ни выбрали?**

<details><summary>Ответ</summary>

Судья (валидация выходов: тесты, схемы, чекеры), лимиты (раунды, сообщения,
токены, таймауты) и ветка честного отказа с журналом. Всё остальное — роли,
маршруты, память — можно делегировать фреймворку.
</details>

## Финальная практика

Это ваш выпускной проект модуля — сначала пробуйте сами. Эталоны всех трёх заданий
лежат в `agent-team/` (сверяться после попытки; `pytest` там гоняется без Ollama).
По возрастанию сложности:

1. **Полный автомат на LangGraph.** Перенесите из 7.2 недостающие узлы:
   аналитика, трассировщика и эскалацию с чистого листа (подсказка: флаг
   `escalated` в состоянии + ещё одно условное ребро). Сравните журнал прогона
   с уроком 7.2 — поведение должно совпасть узел в узел.
   *Эталон:* `team/graph.py` — семь узлов (analyst, developer, tester, tracer, fixer,
   escalate, finish), `route()` с флагом `escalated`; `team/imperative.py` — тот же
   `run_team` из 7.2 на общих промптах; `tests/test_imperative_vs_graph.py` сравнивает
   журналы обоих узел в узел на пяти сценариях (REJECTED — те же 11 вызовов, что в 7.2).
   Живьём: `python scripts/run_graph.py --compare`.

2. **Supervisor по-настоящему.** Соберите в AutoGen `SelectorGroupChat` из трёх
   исполнителей (аналитик, разработчик, критик), где следующего выбирает LLM.
   Посчитайте цену выбора: сколько токенов уходит на сами решения «кто говорит
   следующим» и как часто 3B выбирает не того (это и есть цена supervisor-паттерна
   из 7.1, теперь в цифрах).
   *Эталон:* `team/supervisor.py` — `SelectorGroupChat` с русским `selector_prompt`,
   отдельный клиент-счётчик для selector'а, `expected_next()` как эталонная политика
   конвейера (по ней считаются «не те» выборы и переспросы), режимы `selector | policy |
   roundrobin` и критик-судья (`--critic tests`) — участник чата без LLM.
   Живьём: `python scripts/run_supervisor.py --mode all --critic tests`.

3. **Капстоун.** Команда агентов для задачи из вашей практики: свой судья-код
   (тесты, схема, чекеры — что проверяемо), лимиты, журнал, ветка «нужен человек».
   Прогоните тот же eval на одиночном агенте и честно решите, окупилась ли
   команда. Финальный штрих курса: заверните результат в `ollama-proxy` из
   урока 5.7 — прод-обвязка у вас уже есть.
   *Эталон:* `team/triage.py` — триаж тикетов «Пингвин.Хост»: судья-код из трёх проверок
   (pydantic-схема, заземление цитат, правила R1–R5), eval на 12 тикетах с золотой
   разметкой (которая сама проходит судью — это тест), три конфигурации `solo | solo+fix |
   team` и честный вердикт `paid_off()`; `TEAM_BACKEND=proxy` гонит те же вызовы через
   `ollama-proxy` (токен, лимит, потолок, метрики). Живьём: `python scripts/run_triage.py`.

## 🎓 Итоги модуля 7

- **Команда — это специализация плюс оркестрация.** Роли с узкими промптами
  и своими бюджетами; оркестратор — код, который двигает работу и принимает
  решения (7.1, 7.2).
- **Судья превыше всего.** Тесты в песочнице для кода, чеклисты для решений,
  читаемые отчёты как интерфейс. LLM не судит то, что может судить код (7.2).
- **Ограничения 3B — не приговор, а требования к архитектуре:** жёсткие форматы,
  валидация выходов, лимиты, эскалация, честный REJECTED. Человек с журналом —
  полноправная ветка системы (7.2).
- **Фреймворки — четыре парадигмы одной работы:** декларативная (CrewAI),
  графовая (LangGraph), диалоговая (AutoGen) и ваш код. Выбор — от задачи;
  судья, лимиты и отказ всегда остаются вашими (7.3, 7.4).
- Цена всегда рядом: ×N вызовов, обвязка в токенах, секунды латентности —
  и таблица 7.1, с которой всё началось.

**Дальше — модуль 8, финальный проект курса:** AI-ассистент разработчика —
полноценное FastAPI-приложение, где встретится всё: RAG (модули 2–4), локальная
модель и прод-обвязка (модуль 5), и решения, принятые по-взрослому: с eval,
лимитами и честными отказами.

## 📚 Что почитать

- [LangGraph: multi-agent](https://langchain-ai.github.io/langgraph/concepts/multi_agent/) — supervisor, swarm и команды на графах
- [AutoGen AgentChat](https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/index.html) — команды, терминации, selector
- [AG2](https://docs.ag2.ai/) — форк классического AutoGen API
- [Anthropic: Building effective agents](https://www.anthropic.com/research/building-effective-agents) — рамка, в которой стоит держать весь этот зоопарк
